# putEMG — Within-Subject Deep Learning Evaluation

Replicates the **putEMG paper's evaluation protocol** (Kaczmarek et al., 2019) using
the same CNN/TCN architectures from the generalizable pipeline.

| # | Model | Description |
|---|-------|-------------|
| 1 | **EEGNet** | Depthwise separable CNN, original baseline |
| 2 | **ShallowConvNet** | Temporal + spatial conv, square/log nonlinearity |
| 3 | **DeepConvNet** | 4 stacked conv blocks, increasing filter depth |
| 4 | **CNN_LSTM** | Spatial CNN → temporal pooling → 2-layer LSTM |
| 5 | **EMG_TCN** | Spatial mixing + 4 dilated temporal conv blocks |

**Within-subject split (per fold):**
- Each subject's repetitions are split into `N_FOLDS` stratified folds
- **Train** — 2 folds (~187 reps) | **Test** — 1 fold (~93 reps)
- Rotate `N_FOLDS` times per subject; report per-subject mean → grand mean ± std
- Early stopping monitors **training loss** (no dev set — per-subject data is limited)

**Benchmark:** putEMG paper SVM/RMS ~90% (same within-subject protocol)

> **Prerequisites**: Run `data_preprocessing/driver.ipynb` first to generate per-subject `.mat` files.

In [8]:
# -- Main libraries --
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader


# -- Shared utilities --
_PUBLIC = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', 'public'))
if _PUBLIC not in sys.path:
    sys.path.insert(0, _PUBLIC)

# -- Created functions -- 
import deep_learning_models
from emg_loader import BCIDataset, load_all_subjects, make_within_subject_loaders


# -- Config --
DATA_DIR   = '/Volumes/KRIS/data/UG_per_subject'
BATCH_SIZE = 16

subjects = load_all_subjects(DATA_DIR)

Loading 44 subject file(s) from: /Volumes/KRIS/data/UG_per_subject

  → emg_gestures_03_U.mat  (280 samples)
  → emg_gestures_04_U.mat  (280 samples)
  → emg_gestures_05_U.mat  (280 samples)
  → emg_gestures_06_U.mat  (280 samples)
  → emg_gestures_07_U.mat  (280 samples)
  → emg_gestures_08_U.mat  (280 samples)
  → emg_gestures_09_U.mat  (280 samples)
  → emg_gestures_10_U.mat  (280 samples)
  → emg_gestures_11_U.mat  (280 samples)
  → emg_gestures_12_U.mat  (280 samples)
  → emg_gestures_13_U.mat  (280 samples)
  → emg_gestures_14_U.mat  (280 samples)
  → emg_gestures_15_U.mat  (280 samples)
  → emg_gestures_16_U.mat  (280 samples)
  → emg_gestures_17_U.mat  (280 samples)
  → emg_gestures_18_U.mat  (280 samples)
  → emg_gestures_19_U.mat  (280 samples)
  → emg_gestures_20_U.mat  (280 samples)
  → emg_gestures_22_U.mat  (280 samples)
  → emg_gestures_23_U.mat  (280 samples)
  → emg_gestures_24_U.mat  (280 samples)
  → emg_gestures_25_U.mat  (280 samples)
  → emg_gestures_26_U.mat  (28

In [9]:
# ── Training & evaluation utilities ──────────────────────────────────────────
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def train(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out  = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)


def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            pred = model(X).argmax(dim=1)
            correct += (pred == y).sum().item()
            total   += y.size(0)
    return correct / total


def evaluateFinal(model, test_loader, device):
    model.eval()
    correct, total = 0, 0
    actual    = torch.tensor([], dtype=torch.int64)
    predicted = torch.tensor([], dtype=torch.int64)
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            pred = model(X).argmax(dim=1)
            correct   += (pred == y).sum().item()
            total     += y.size(0)
            actual    = torch.cat((actual,    y.cpu()),    dim=0)
            predicted = torch.cat((predicted, pred.cpu()), dim=0)

        print(np.unique(predicted.numpy(), return_counts=True))
        cm = confusion_matrix(actual.numpy(), predicted.numpy())
        ConfusionMatrixDisplay(cm).plot()

    print(f'The accuracy for the test is {(correct / total) * 100:.2f}%')
    return correct / total

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [11]:
# -- Config --
N_FOLDS     = 3     # folds per subject — matches putEMG paper protocol
N_SUBJECTS  = None     # 
SEED        = 42    # random seed for stratified fold generation
PATIENCE    = 5     # early-stopping patience (epochs without improvement)
MAX_EPOCHS  = 30
MIN_DELTA   = 0.001 # minimum improvement to reset patience
DROPOUT     = 0.1
LR          = 1e-3
WEIGHTS_DIR = 'weights'

---
## Training

Each model is trained independently **per subject** with:
- **Stratified 3-fold CV** — class balance preserved across every fold
- **Early stopping** — stops when training loss doesn't improve by `MIN_DELTA` for `PATIENCE` epochs
- **ReduceLROnPlateau** — halves the LR when training loss plateaus for 3 epochs
- **Best-state checkpointing** — restores the lowest-loss weights before test evaluation

Results are collected per fold. After all folds, mean accuracy is reported per subject → grand mean ± std across all subjects.

In [12]:
model_registry = {
    'EEGNet':         lambda: deep_learning_models.EEGNet(dropout_rate=DROPOUT),
    'ShallowConvNet': lambda: deep_learning_models.ShallowConvNet(dropout_rate=DROPOUT),
    # 'DeepConvNet':    lambda: deep_learning_models.DeepConvNet(dropout_rate=DROPOUT),
    # 'CNN_LSTM':       lambda: deep_learning_models.CNN_LSTM(dropout_rate=DROPOUT),
    'EMG_TCN':        lambda: deep_learning_models.EMG_TCN(dropout_rate=DROPOUT),
}

In [13]:
subject_results = {}          # {subject_name: {model_name: [fold accs]}}

run_subjects = subjects[:N_SUBJECTS] if N_SUBJECTS else subjects
print(f'Subjects : {len(run_subjects)}')
print(f'Folds    : {N_FOLDS} per subject')
print(f'Models   : {list(model_registry.keys())}')
print(f'Total    : {len(run_subjects) * len(model_registry)} subject-model pairs')

Subjects : 44
Folds    : 3 per subject
Models   : ['EEGNet', 'ShallowConvNet', 'EMG_TCN']
Total    : 132 subject-model pairs


In [14]:
for sub_idx, (sub_name, X_sub, y_sub) in enumerate(run_subjects):

    sub_id = sub_name.split('_')[2]                     # 'emg_gestures_03_U.mat' → '03'

    print(f"\n{'#'*60}")
    print(f"  Subject {sub_idx + 1}/{len(run_subjects)}  —  subject_{sub_id}")
    print(f"{'#'*60}")

    subject_results.setdefault(sub_name, {})

    for name, build_fn in model_registry.items():

        model_dir = os.path.join(WEIGHTS_DIR, name)
        save_path = os.path.join(model_dir, f'subject_{sub_id}.pt')

        # -- Skip if already trained --
        if os.path.exists(save_path):
            checkpoint = torch.load(save_path, map_location='cpu')
            subject_results[sub_name][name] = checkpoint['fold_accs']
            print(f"  [{name}] subject_{sub_id} already trained "
                  f"(mean={np.mean(checkpoint['fold_accs'])*100:.2f}%) — skipping.")
            continue

        print(f"\n{'='*50}")
        print(f"  Training: {name}")
        print(f"{'='*50}")

        fold_accs   = []
        fold_states = []

        for fold_idx in range(N_FOLDS):

            train_loader, test_loader = make_within_subject_loaders(
                X_sub, y_sub,
                n_folds=N_FOLDS, test_fold_idx=fold_idx,
                batch_size=BATCH_SIZE, seed=SEED,
            )

            model     = build_fn().to(device)
            optimizer = optim.Adam(model.parameters(), lr=LR)
            scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5,
                                          patience=3, min_lr=1e-6)
            criterion = nn.CrossEntropyLoss()

            best_loss  = float('inf')
            best_state = None
            bad_epochs = 0

            for epoch in range(MAX_EPOCHS):
                tr_loss = train(model, train_loader, criterion, optimizer, device)
                curr_lr = optimizer.param_groups[0]['lr']

                if tr_loss < best_loss - MIN_DELTA:
                    best_loss  = tr_loss
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                    bad_epochs = 0
                else:
                    bad_epochs += 1

                scheduler.step(tr_loss)

                print(f"  Epoch {epoch+1:3d}: loss={tr_loss:.4f}  "
                      f"best={best_loss:.4f}  lr={curr_lr:.2e}")

                if bad_epochs >= PATIENCE:
                    print(f"  Early stopping at epoch {epoch+1}.")
                    break

            model.load_state_dict(best_state)

            print(f"\n  --- {name} | subject_{sub_id}, Fold {fold_idx + 1} — Test Evaluation ---")
            test_acc = evaluate(model, test_loader, device)
            print(f"  Test accuracy: {test_acc * 100:.2f}%")

            fold_accs.append(test_acc)
            fold_states.append(best_state)

        # -- Save best-fold weights for this subject-model pair --
        best_fold_idx = int(np.argmax(fold_accs))
        subject_results[sub_name][name] = fold_accs

        os.makedirs(model_dir, exist_ok=True)
        torch.save({
            'model_name':   name,
            'subject_id':   f'subject_{sub_id}',
            'subject_file': sub_name,
            'fold_accs':    fold_accs,
            'mean_acc':     float(np.mean(fold_accs)),
            'best_fold':    best_fold_idx,
            'n_folds':      N_FOLDS,
            'dropout':      DROPOUT,
            'state_dict':   fold_states[best_fold_idx],
        }, save_path)

        print(f"\n  Saved → {save_path}")
        print(f"  Fold accs : {[f'{a*100:.2f}%' for a in fold_accs]}")
        print(f"  Mean acc  : {np.mean(fold_accs)*100:.2f}%")


############################################################
  Subject 1/44  —  subject_03
############################################################
  [EEGNet] subject_03 already trained (mean=98.90%) — skipping.
  [ShallowConvNet] subject_03 already trained (mean=97.49%) — skipping.

  Training: EMG_TCN
    Fold 1/3: train=182  test=98
  Epoch   1: loss=1.3976  best=1.3976  lr=1.00e-03
  Epoch   2: loss=0.5262  best=0.5262  lr=1.00e-03
  Epoch   3: loss=0.2125  best=0.2125  lr=1.00e-03
  Epoch   4: loss=0.1474  best=0.1474  lr=1.00e-03
  Epoch   5: loss=0.1287  best=0.1287  lr=1.00e-03
  Epoch   6: loss=0.0976  best=0.0976  lr=1.00e-03
  Epoch   7: loss=0.0672  best=0.0672  lr=1.00e-03
  Epoch   8: loss=0.0666  best=0.0672  lr=1.00e-03
  Epoch   9: loss=0.0538  best=0.0538  lr=1.00e-03
  Epoch  10: loss=0.0754  best=0.0538  lr=1.00e-03
  Epoch  11: loss=0.0487  best=0.0487  lr=1.00e-03
  Epoch  12: loss=0.0535  best=0.0487  lr=1.00e-03
  Epoch  13: loss=0.0575  best=0.0487  lr=1.0

KeyboardInterrupt: 

---
## Results Summary

In [15]:
print(f"\n{'Model':<18} {'Subjects':>8}  {'Mean Acc':>10}  {'Std':>8}")
print('-' * 55)
for name in model_registry:
    sub_means = [
        np.mean(subject_results[s][name]) * 100
        for s in subject_results
        if subject_results[s][name]
    ]
    if sub_means:
        mean, std = np.mean(sub_means), np.std(sub_means)
        print(f"{name:<18} {len(sub_means):>8}  {mean:>9.2f}%  {std:>7.2f}%")
    else:
        print(f"{name:<18} {'—':>8}  {'—':>10}  {'—':>8}")

# Bar chart — grand mean accuracy per model
names_with_results = [
    n for n in model_registry
    if any(subject_results[s][n] for s in subject_results)
]
means = [
    np.mean([np.mean(subject_results[s][n]) * 100 for s in subject_results if subject_results[s][n]])
    for n in names_with_results
]
stds = [
    np.std([np.mean(subject_results[s][n]) * 100 for s in subject_results if subject_results[s][n]])
    for n in names_with_results
]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(names_with_results, means, yerr=stds, capsize=5, color='steelblue')
ax.axhline(y=90, color='red', linestyle='--', alpha=0.6, label='putEMG paper (~90%)')
ax.set_ylim(0, 110)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title(f'Within-Subject Results ({N_FOLDS} fold(s) × {len(subject_results)} subject(s)) — Mean ± Std')
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f'{m:.1f}%', ha='center', va='bottom', fontsize=10)
ax.legend()
plt.tight_layout()
plt.show()


Model              Subjects    Mean Acc       Std
-------------------------------------------------------
EEGNet                   38      91.05%     5.56%
ShallowConvNet           38      93.50%     4.09%


KeyError: 'EMG_TCN'

---
## Best Architecture

In [16]:
# Weights were already saved per subject-model pair during training.
# This cell identifies the winning architecture by grand mean across subjects.

model_means = {}
for name in model_registry:
    sub_means = [
        np.mean(subject_results[s][name]) * 100
        for s in subject_results
        if name in subject_results[s] and subject_results[s][name]
    ]
    if sub_means:
        model_means[name] = (np.mean(sub_means), np.std(sub_means), len(sub_means))

print(f"\n{'Model':<18} {'Subjects':>8}  {'Mean Acc':>10}  {'Std':>8}")
print('-' * 55)
for name, (mean, std, n) in model_means.items():
    print(f"{name:<18} {n:>8}  {mean:>9.2f}%  {std:>7.2f}%")

if model_means:
    best_architecture = max(model_means, key=lambda n: model_means[n][0])
    best_mean         = model_means[best_architecture][0]
    print(f"\n→ Best architecture : {best_architecture}  ({best_mean:.2f}%)")
    print(f"  Weights saved in  : {os.path.join(WEIGHTS_DIR, best_architecture)}/")


Model              Subjects    Mean Acc       Std
-------------------------------------------------------
EEGNet                   38      91.05%     5.56%
ShallowConvNet           38      93.50%     4.09%
EMG_TCN                  37      97.47%     2.78%

→ Best architecture : EMG_TCN  (97.47%)
  Weights saved in  : weights/EMG_TCN/


---
## Save Report

In [17]:
from datetime import datetime

os.makedirs(WEIGHTS_DIR, exist_ok=True)
report_path = os.path.join(WEIGHTS_DIR, 'report.txt')

# ── Compute per-model grand stats ─────────────────────────────────────────────
model_stats = {}
for name in model_registry:
    sub_means = [
        np.mean(subject_results[s][name]) * 100
        for s in subject_results
        if name in subject_results[s] and subject_results[s][name]
    ]
    if sub_means:
        model_stats[name] = (np.mean(sub_means), np.std(sub_means), len(sub_means))

best_architecture = max(model_stats, key=lambda n: model_stats[n][0]) if model_stats else '—'

# ── Build report string ───────────────────────────────────────────────────────
W  = 80
sep  = '=' * W
thin = '-' * W
now  = datetime.now().strftime('%Y-%m-%d  %H:%M:%S')

lines = []
lines.append(sep)
lines.append('  putEMG — Within-Subject Deep Learning Evaluation Report')
lines.append(f'  Generated : {now}')
lines.append(sep)
lines.append('')
lines.append('Config')
lines.append(thin)
lines.append(f'  Models tested      : {", ".join(model_registry.keys())}')
lines.append(f'  Subjects evaluated : {len(subject_results)}')
lines.append(f'  Folds per subject  : {N_FOLDS}')
lines.append(f'  Dropout            : {DROPOUT}')
lines.append(f'  Max epochs         : {MAX_EPOCHS}')
lines.append(f'  Early stop         : patience={PATIENCE}, min_delta={MIN_DELTA}')
lines.append(f'  Seed               : {SEED}')
lines.append('')

# ── Model summary ─────────────────────────────────────────────────────────────
lines.append(thin)
lines.append('Model Summary  (grand mean ± std across subjects)')
lines.append(thin)
lines.append(f'  {"Model":<18}  {"Subjects":>8}  {"Mean Acc":>10}  {"Std":>8}')
lines.append(f'  {"-"*18}  {"-"*8}  {"-"*10}  {"-"*8}')
for name, (mean, std, n) in sorted(model_stats.items(), key=lambda x: -x[1][0]):
    marker = '  ←' if name == best_architecture else ''
    lines.append(f'  {name:<18}  {n:>8}  {mean:>9.2f}%  {std:>7.2f}%{marker}')
lines.append('')
lines.append(f'  → Best architecture : {best_architecture}  '
             f'({model_stats[best_architecture][0]:.2f}%)')
lines.append(f'  → Weights saved in  : {os.path.join(WEIGHTS_DIR, best_architecture)}/')
lines.append('')

# ── Per-subject breakdown ─────────────────────────────────────────────────────
model_names = list(model_registry.keys())
col = 10

lines.append(thin)
lines.append('Per-Subject Results')
lines.append(thin)
header = f'  {"Subject":<12}' + ''.join(f'{n[:col]:>{col+2}}' for n in model_names)
header += f'  {"Best Model":<16}  {"Best Acc":>8}'
lines.append(header)
lines.append(f'  {"-"*12}' + ''.join(f'  {"-"*col}' for _ in model_names)
             + f'  {"-"*16}  {"-"*8}')

for sub_name, results in sorted(subject_results.items()):
    sub_id = f'subject_{sub_name.split("_")[2]}'
    accs   = {name: np.mean(results[name]) * 100
              for name in model_names if name in results and results[name]}
    if not accs:
        continue
    best_model = max(accs, key=accs.get)
    best_acc   = accs[best_model]
    row = f'  {sub_id:<12}'
    for name in model_names:
        val = f'{accs[name]:.2f}%' if name in accs else '—'
        row += f'  {val:>{col}}'
    row += f'  {best_model:<16}  {best_acc:>7.2f}%'
    lines.append(row)

lines.append('')
lines.append(sep)

report_text = '\n'.join(lines)

with open(report_path, 'w') as f:
    f.write(report_text)

print(report_text)
print(f'\nReport saved → {report_path}')

  putEMG — Within-Subject Deep Learning Evaluation Report
  Generated : 2026-04-19  21:43:31

Config
--------------------------------------------------------------------------------
  Models tested      : EEGNet, ShallowConvNet, EMG_TCN
  Subjects evaluated : 38
  Folds per subject  : 3
  Dropout            : 0.1
  Max epochs         : 30
  Early stop         : patience=5, min_delta=0.001
  Seed               : 42

--------------------------------------------------------------------------------
Model Summary  (grand mean ± std across subjects)
--------------------------------------------------------------------------------
  Model               Subjects    Mean Acc       Std
  ------------------  --------  ----------  --------
  EMG_TCN                   37      97.47%     2.78%  ←
  ShallowConvNet            38      93.50%     4.09%
  EEGNet                    38      91.05%     5.56%

  → Best architecture : EMG_TCN  (97.47%)
  → Weights saved in  : weights/EMG_TCN/

----------------